# 260420_cpi_install_to_login_analysis

| Field | Value |
|-------|-------|
| **Author** | Haewon Yum |
| **Last update** | 2026-04-21 |
| **Objective** | Two analyses: (1) Cross-country install-to-login rate from Netmarble CPI campaigns — show KOR has a comparably high rate and competitive implied login CPA (= CPI ÷ rate), supporting the case for CPI campaigns in KOR. (2) Attributed vs unattributed comparison — validate that the attributed rate is genuine by showing it's consistent with the unattributed baseline for the same titles and countries. |
| **Scope** | All Netmarble titles with CPI campaigns (last 24 months); iOS + Android analyzed at OS level; KOR always shown + other countries ≥50 installs; D1 login window (configurable) |
| **Out of scope** | CPA/ROAS/RE campaigns; titles without CPI campaigns; post-D1 login rate |
| **Key tables** | `moloco-ae-view.athena.fact_dsp_core`, `moloco-dsp-data-view.postback.pb` (authoritative), `focal-elf-631.prod_stream_view.cv` |
| **Additional context** | Netmarble's internal KPI is login CPA — they run CPA (login) campaigns. Hypothesis 1: KOR install-to-login rate is high enough that implied login CPA from CPI is competitive. Hypothesis 2: Attributed rate ≈ unattributed rate → CPI acquires users with genuine login intent. Unattributed traffic may include other paid channels (not a clean organic baseline). |

In [32]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
client = bigquery.Client(project="moloco-ods")

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'\u2705 {label}: {len(df)} rows')
    return df

CHART_DIR = os.path.join(os.getcwd(), 'charts')  # charts saved to charts/ subfolder
os.makedirs(CHART_DIR, exist_ok=True)

In [34]:
from datetime import date, timedelta

ANALYSIS_DATE     = str(date.today())
START_DATE        = str(date.today() - timedelta(days=730))
LOGIN_WINDOW_DAYS = 1  # D1 login window

# Per-title launch windows — source: SensorTower, verified 2026-04-21
# Keys = store_id format (fact_dsp_core / cv table): numeric for iOS, com.* for Android
# Excluded: com.netmarble.bnsmkr (BnS Revolution, launched 2018)
# pb source: focal-elf-631.df_accesslog.pb (2-year history available)
LAUNCH_WINDOWS = {
    'com.netmarble.sololv':      ('2024-05-03', '2024-06-03'),  # Solo Leveling: Arise (Android)
    '1662742277':                ('2024-05-03', '2024-06-03'),  # Solo Leveling: Arise (iOS)
    'com.netmarble.nanarise':    ('2024-08-13', '2024-09-13'),  # Seven Deadly Sins: IDLE (Android)
    '6469305531':                ('2024-08-13', '2024-09-13'),  # Seven Deadly Sins: IDLE (iOS)
    'com.kabam.knights.legends': ('2024-11-14', '2024-12-14'),  # King Arthur: Legends Rise (Android)
    'com.netmarble.got':         ('2025-05-20', '2025-06-20'),  # Game of Thrones: Kingsroad (Android)
    '6499177365':                ('2025-09-04', '2025-10-04'),  # THE KING OF FIGHTERS AFK (iOS)
    'com.netmarble.tskgb':       ('2025-09-12', '2025-10-12'),  # 세븐나이츠 리버스 (Android)
    '6479595079':                ('2025-09-12', '2025-10-12'),  # 세븐나이츠 리버스 (iOS)
}

# BQ partition bounds: earliest launch → latest window end + 1d D1 buffer
PARTITION_START = min(s for s, _ in LAUNCH_WINDOWS.values())
PARTITION_END   = str(max(date.fromisoformat(e) for _, e in LAUNCH_WINDOWS.values())
                      + timedelta(days=1))

SCOPE_BUNDLES = list(LAUNCH_WINDOWS.keys())

print(f'Analysis period: {START_DATE} → {ANALYSIS_DATE}')
print(f'Login window: D{LOGIN_WINDOW_DAYS}')
print(f'Scope: {len(SCOPE_BUNDLES)} bundles across 6 titles')
print(f'Partition window: {PARTITION_START} → {PARTITION_END}')

Analysis period: 2024-04-21 → 2026-04-21
Login window: D1
Scope: 9 bundles across 6 titles
Partition window: 2024-05-03 → 2025-10-13


## Section 0 - Netmarble CPI Campaign Discovery (Last 24 Months)

Identify all Netmarble CPI campaigns across all titles and OS. Output: campaign IDs, app name, bundle, MMP, OS, spend, installs, date range. Flag titles with <100 total installs as insufficient for reliable rate calculation. Campaign IDs from this section scope Sections 1\u20134.

In [35]:
# Per-bundle date filter for fact_dsp_core (date_utc column, store_id format)
_s0_clauses = ' OR\n    '.join(
    f"(product.app_market_bundle = '{b}' AND date_utc BETWEEN '{s}' AND '{e}')"
    for b, (s, e) in LAUNCH_WINDOWS.items()
)

query_0 = f"""
WITH daily_rows AS (
  SELECT
    product.app_name                                            AS app_name,
    product.app_market_bundle                                   AS bundle_id,
    campaign.os                                                 AS os,
    campaign.country                                            AS country,
    campaign_id,
    advertiser.title                                            AS advertiser_title,
    date_utc,
    gross_spend_usd,
    installs
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE date_utc BETWEEN '{PARTITION_START}' AND '{PARTITION_END}'
    AND LOWER(advertiser.title) LIKE '%netmarble%'
    AND campaign.goal = 'OPTIMIZE_CPI_FOR_APP_UA'
    AND ({_s0_clauses})
),

campaign_summary AS (
  SELECT
    app_name,
    bundle_id,
    os,
    country,
    campaign_id,
    advertiser_title,
    SUM(gross_spend_usd)                                        AS total_spend_usd,
    SUM(installs)                                               AS total_installs,
    SAFE_DIVIDE(SUM(gross_spend_usd), NULLIF(SUM(installs), 0)) AS cpi_usd,
    MIN(date_utc)                                               AS first_date,
    MAX(date_utc)                                               AS last_date,
    COUNT(DISTINCT date_utc)                                    AS active_days
  FROM daily_rows
  GROUP BY 1, 2, 3, 4, 5, 6
),

with_flags AS (
  SELECT
    *,
    COUNT(DISTINCT campaign_id) OVER (
      PARTITION BY app_name, bundle_id, os, country
    )                                                           AS campaign_count,
    IF(total_installs < 50, TRUE, FALSE)                        AS low_sample_flag
  FROM campaign_summary
)

SELECT
  app_name,
  bundle_id,
  os,
  country,
  campaign_id,
  advertiser_title,
  ROUND(total_spend_usd, 2)   AS total_spend_usd,
  total_installs,
  ROUND(cpi_usd, 2)           AS cpi_usd,
  first_date,
  last_date,
  active_days,
  campaign_count,
  low_sample_flag
FROM with_flags
ORDER BY app_name, os, total_installs DESC
"""
df_0 = run_query(query_0, label='Netmarble CPI Campaign Discovery')
df_0

✅ Netmarble CPI Campaign Discovery: 106 rows


,app_name,bundle_id,os,country,campaign_id,advertiser_title,total_spend_usd,total_installs,cpi_usd,first_date,last_date,active_days,campaign_count,low_sample_flag
0,Game of Thrones: Kingsroad,com.netmarble.got,ANDROID,USA,XclglCmytTd3oZdu,netmarble,26144.150000000,7223,3.620000000,2025-05-22,2025-06-20,30,1,False
1,Game of Thrones: Kingsroad,com.netmarble.got,ANDROID,SAU,uGxkpqkeQh99T989,netmarble,5598.330000000,4616,1.210000000,2025-05-22,2025-06-20,30,1,False
2,Game of Thrones: Kingsroad,com.netmarble.got,ANDROID,FRA,taPHzWZoXlnJFVuL,netmarble,11598.860000000,4338,2.670000000,2025-05-22,2025-06-20,30,2,False
3,Game of Thrones: Kingsroad,com.netmarble.got,ANDROID,DEU,dxy6ywiE5NH8NALH,netmarble,11644.680000000,4036,2.890000000,2025-05-22,2025-06-20,30,2,False
4,Game of Thrones: Kingsroad,com.netmarble.got,ANDROID,ITA,uGxkpqkeQh99T989,netmarble,2903.890000000,2627,1.110000000,2025-05-22,2025-06-20,30,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,The Seven Deadly Sins: IDLE,6469305531,IOS,USA,Ej0kjskQi5OKCnbM,netmarble,5564.100000000,407,13.670000000,2024-09-10,2024-09-13,4,1,False
102,The Seven Deadly Sins: IDLE,6469305531,IOS,None,kXpSqUDwIFgXsqO3,netmarble,0E-9,0,None,2024-08-17,2024-09-13,28,2,True
103,The Seven Deadly Sins: IDLE,6469305531,IOS,None,Ej0kjskQi5OKCnbM,netmarble,0E-9,0,None,2024-09-11,2024-09-13,3,2,True
104,세븐나이츠 리버스,6479595079,IOS,KOR,sYGRrSSi1eD0N0Gq,netmarble,50845.080000000,5192,9.790000000,2025-09-12,2025-10-12,31,1,False


In [56]:
# Build filters and title inventory for downstream sections
CAMPAIGN_IDS = df_0['campaign_id'].unique().tolist()
BUNDLE_IDS   = df_0['bundle_id'].unique().tolist()  # as-is from fact_dsp_core

# cv table: store_id format — Android = com.xxx, iOS = numeric (no 'id' prefix)
# fact_dsp_core iOS bundles are already numeric (e.g. '6499177365'), so no stripping needed
STORE_IDS = BUNDLE_IDS  # already in cv store_id format

# pb table: MMP bundle format — iOS needs 'id' prefix for numeric bundle IDs
PB_BUNDLES = [
    f'id{b}' if b.isdigit() else b
    for b in BUNDLE_IDS
]

CAMPAIGN_IDS_SQL = "('" + "', '".join(CAMPAIGN_IDS) + "')"
BUNDLE_IDS_SQL   = "('" + "', '".join(PB_BUNDLES)   + "')"   # pb table (id-prefix for iOS)
STORE_IDS_SQL    = "('" + "', '".join(STORE_IDS)    + "')"   # cv table (no id-prefix)

print(f'{df_0["campaign_id"].nunique()} campaigns across {df_0["app_name"].nunique()} titles')
print(f'Bundles (pb/MMP format):  {PB_BUNDLES}')
print(f'Store IDs (cv format):    {STORE_IDS}')
print()
print('Spend summary by title × OS:')
display(
    df_0.groupby(['app_name', 'os'], as_index=False)
    .agg(total_spend_usd=('total_spend_usd', 'sum'),
         total_installs=('total_installs', 'sum'),
         avg_cpi_usd=('cpi_usd', 'mean'),
         low_sample_countries=('low_sample_flag', 'sum'))
    .round(2)
    .sort_values('total_installs', ascending=False)
)

# store_id → app_name for chart labels (derived from Section 0)
TITLE_LOOKUP = {
    row['bundle_id']: row['app_name']
    for _, row in df_0[['bundle_id', 'app_name']].drop_duplicates().iterrows()
}

31 campaigns across 6 titles
Bundles (pb/MMP format):  ['com.netmarble.got', 'com.kabam.knights.legends', 'com.netmarble.sololv', 'id6499177365', 'com.netmarble.nanarise', 'id6469305531', 'id6479595079']
Store IDs (cv format):    ['com.netmarble.got', 'com.kabam.knights.legends', 'com.netmarble.sololv', '6499177365', 'com.netmarble.nanarise', '6469305531', '6479595079']

Spend summary by title × OS:


,app_name,os,total_spend_usd,total_installs,avg_cpi_usd,low_sample_countries
2,Solo Leveling:Arise,ANDROID,499286.210000000,578067,1.01697,2
4,The Seven Deadly Sins: IDLE,ANDROID,156023.150000000,55912,2.868095,3
0,Game of Thrones: Kingsroad,ANDROID,69079.200000000,30230,1.541923,0
3,THE KING OF FIGHTERS AFK,IOS,98771.770000000,11312,7.388571,5
1,King Arthur: Legends Rise,ANDROID,22234.850000000,7003,3.0875,1
6,세븐나이츠 리버스,IOS,50845.080000000,5192,9.79,1
5,The Seven Deadly Sins: IDLE,IOS,33319.720000000,3179,11.84,2


## Section 1 - Login Event Discovery per Title

For each title from Section 0, identify the primary first-login event name from the pb table. Login event names vary by MMP and advertiser config — never hardcode 'login'. Output: title --> candidate login event names with postback count. Manual review step: confirm the primary login event per title before proceeding to Section 2.

In [46]:
# # Login event discovery — using focal-elf-631.df_accesslog.pb (2-year history)
# # Covers full PARTITION_START → PARTITION_END to capture all launch windows
# query_1 = f"""
# SELECT
#   app.bundle                  AS bundle,
#   event.name                  AS event_name,
#   COUNT(*)                    AS postback_count
# FROM `focal-elf-631.df_accesslog.pb`
# WHERE
#   DATE(timestamp) BETWEEN '{PARTITION_START}' AND '{PARTITION_END}'
#   AND app.bundle IN {BUNDLE_IDS_SQL}
#   AND LOWER(event.name) LIKE '%login%'
#   AND attribution.attributed = TRUE
# GROUP BY app.bundle, event.name
# ORDER BY app.bundle, postback_count DESC
# """
# df_1 = run_query(query_1, label='Login Event Discovery')
# df_1

In [47]:
# ── Login event names confirmed via focal-elf-631.prod_stream_view.cv (2026-04-21) ──
# Keys = MMP bundle (id-prefix for iOS); values = event_name string.
LOGIN_EVENTS = {
    'com.netmarble.sololv':      'login',          # confirmed
    'id1662742277':              'login',          # confirmed
    'com.netmarble.nanarise':    'login',          # confirmed
    'id6469305531':              'login',          # confirmed
    'com.kabam.knights.legends': 'login',          # confirmed
    'com.netmarble.got':         'login',          # confirmed
    'com.netmarble.tskgb':       'login_1st',      # confirmed
    'id6479595079':              'login_1st',      # confirmed
    'id6499177365':              'login_complete',  # confirmed
}

LOGIN_EVENT_NAMES = list(set(e.lower() for e in LOGIN_EVENTS.values()))

if LOGIN_EVENT_NAMES:
    LOGIN_EVENTS_SQL = "('" + "', '".join(LOGIN_EVENT_NAMES) + "')"
    print('✅ Login events confirmed:', LOGIN_EVENT_NAMES)
    print('Proceed to Section 2.')
else:
    print('⚠️  Review df_1 above and populate LOGIN_EVENTS before running Sections 2–3.')

✅ Login events confirmed: ['login_complete', 'login_1st', 'login']
Proceed to Section 2.


## Section 2 - Attributed Install-to-Login Rate (CPI Campaigns)

For each CPI campaign title, compute the fraction of attributed installs that generate a login event within D1. Group by title \u00d7 OS \u00d7 country. KOR always shown; other countries included if \u226550 installs. Output: install count, login count, install-to-login rate (%), CPI, implied login CPA = CPI \u00f7 rate.

In [48]:
# Per-bundle install filter for cv table (api.product.app.store_id = store_id format, same as LAUNCH_WINDOWS keys)
_cv_clauses = ' OR\n    '.join(
    f"(api.product.app.store_id = '{b}' AND DATE(cv.install_at_pb) BETWEEN '{s}' AND '{e}')"
    for b, (s, e) in LAUNCH_WINDOWS.items()
)

# Outer partition covers full launch window + 1d D1 buffer; per-bundle filter tightens it further
query_2 = f"""
WITH attributed_events AS (
  SELECT
    api.product.app.store_id    AS store_id,
    req.device.os               AS os,
    req.device.geo.country      AS country,
    bid.mtid                    AS mtid,
    cv.event_pb                 AS event_name,
    cv.install_at_pb            AS install_at,
    timestamp                   AS event_ts
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE
    DATE(timestamp) BETWEEN '{PARTITION_START}' AND '{PARTITION_END}'
    AND api.product.app.store_id IN {STORE_IDS_SQL}
    AND api.campaign.id IN {CAMPAIGN_IDS_SQL}
    AND ({_cv_clauses})
    AND (
      cv.event_pb = 'install'
      OR LOWER(cv.event_pb) IN {LOGIN_EVENTS_SQL}
    )
),

per_user AS (
  SELECT
    store_id,
    os,
    country,
    mtid,
    MAX(CASE WHEN event_name = 'install' THEN 1 ELSE 0 END)   AS has_install,
    MIN(CASE WHEN event_name = 'install' THEN install_at END)  AS install_at,
    MAX(
      CASE
        WHEN LOWER(event_name) IN {LOGIN_EVENTS_SQL}
         AND TIMESTAMP_DIFF(event_ts, install_at, HOUR) BETWEEN 0 AND 24
        THEN 1 ELSE 0
      END
    )                                                          AS has_d1_login
  FROM attributed_events
  GROUP BY store_id, os, country, mtid
)

SELECT
  store_id,
  os,
  country,
  COUNT(DISTINCT CASE WHEN has_install = 1 THEN mtid END)                AS installs,
  COUNT(DISTINCT CASE WHEN has_install = 1 AND has_d1_login = 1
                      THEN mtid END)                                      AS d1_logins,
  ROUND(
    SAFE_DIVIDE(
      COUNT(DISTINCT CASE WHEN has_install = 1 AND has_d1_login = 1 THEN mtid END),
      COUNT(DISTINCT CASE WHEN has_install = 1 THEN mtid END)
    ), 4
  )                                                                       AS install_to_login_rate
FROM per_user
WHERE has_install = 1
GROUP BY store_id, os, country
HAVING COUNT(DISTINCT CASE WHEN has_install = 1 THEN mtid END) >= 50
   OR country = 'KOR'
ORDER BY store_id, installs DESC
"""
df_2 = run_query(query_2, label='Attributed Install-to-Login Rate')
df_2

✅ Attributed Install-to-Login Rate: 89 rows


,store_id,os,country,installs,d1_logins,install_to_login_rate
0,6469305531,IOS,JPN,2772,2735,0.9867
1,6469305531,IOS,USA,407,389,0.9558
2,6479595079,IOS,KOR,5192,4007,0.7718
3,6499177365,IOS,JPN,4565,4399,0.9636
4,6499177365,IOS,KOR,2776,2775,0.9996
...,...,...,...,...,...,...
84,com.netmarble.sololv,ANDROID,AUS,366,320,0.8743
85,com.netmarble.sololv,ANDROID,MAC,289,236,0.8166
86,com.netmarble.sololv,ANDROID,CHE,273,228,0.8352
87,com.netmarble.sololv,ANDROID,NZL,219,182,0.8311


In [57]:
# Join CPI from Section 0 (country-level mean CPI) and compute implied login CPA
cpi_lookup = (
    df_0.assign(
        store_id=df_0['bundle_id'].str.replace(r'^id(\d+)$', r'\1', regex=True)
    )
    .groupby(['store_id', 'os', 'country'], as_index=False)
    .agg(cpi_usd=('cpi_usd', 'mean'))
)

df_2e = df_2.merge(cpi_lookup, on=['store_id', 'os', 'country'], how='left')

# Coerce to float — BQ may return NUMERIC/DECIMAL as Python Decimal (object dtype)
df_2e['install_to_login_rate'] = pd.to_numeric(df_2e['install_to_login_rate'], errors='coerce')
df_2e['cpi_usd']               = pd.to_numeric(df_2e['cpi_usd'],               errors='coerce')

df_2e['install_to_login_pct']   = (df_2e['install_to_login_rate'] * 100).round(1)
df_2e['implied_login_cpa_usd']  = (df_2e['cpi_usd'] / df_2e['install_to_login_rate']).round(2)

for (store_id, os_val), grp in df_2e.groupby(['store_id', 'os']):
    grp = grp.sort_values('install_to_login_pct', ascending=False).copy()
    app_name  = TITLE_LOOKUP.get(store_id, store_id)
    title_str = f'{app_name} ({store_id}) — {os_val}'

    fig = go.Figure(go.Bar(
        x=grp['country'],
        y=grp['install_to_login_pct'],
        marker_color=['crimson' if c == 'KOR' else '#4C78A8' for c in grp['country']],
        text=grp['install_to_login_pct'].apply(lambda v: f'{v:.1f}%'),
        textposition='outside',
    ))
    fig.update_layout(
        title=f'{title_str} — Attributed Install-to-Login Rate by Country (D{LOGIN_WINDOW_DAYS})',
        xaxis_title='Country', yaxis_title='Install-to-Login Rate (%)',
        height=420, template='plotly_white'
    )
    fig.write_image(os.path.join(CHART_DIR, f'attr_i2l_rate_{store_id}_{os_val}.png'), scale=2)
    fig.show()

## Section 3 - Unattributed Install-to-Login Rate (Benchmark)

Compute unattributed baseline install-to-login rate for the same titles, OS, and countries. Unattributed traffic includes users not attributed to Moloco (may include other paid channels, not purely organic). Same D1 window and login event definitions from Section 1.

In [50]:
# Unattributed baseline — 10% deterministic device sample via FARM_FINGERPRINT(idfa).
# Purpose: directional comparison only (attributed rate ≈ unattributed rate?), not exact measurement.
# Sampling is unbiased: each IDFA has equal probability of inclusion regardless of event count.
_pb_launch = {
    (f'id{b}' if b.isdigit() else b): (s, e)
    for b, (s, e) in LAUNCH_WINDOWS.items()
}
_pb_clauses = ' OR\n    '.join(
    f"(app.bundle = '{b}' AND DATE(event.install_at) BETWEEN '{s}' AND '{e}')"
    for b, (s, e) in _pb_launch.items()
)

query_3 = f"""
WITH unattributed_events AS (
  SELECT
    app.bundle                  AS bundle,
    device.os                   AS os,
    device.country              AS country,
    device.idfa                 AS idfa,
    event.name                  AS event_name,
    event.install_at            AS install_at,
    timestamp                   AS event_ts
  FROM `focal-elf-631.df_accesslog.pb`
  WHERE
    DATE(timestamp) BETWEEN '{PARTITION_START}' AND '{PARTITION_END}'
    AND app.bundle IN {BUNDLE_IDS_SQL}
    AND attribution.attributed = FALSE
    AND ({_pb_clauses})
    AND (
      LOWER(event.name) = 'install'
      OR LOWER(event.name) IN {LOGIN_EVENTS_SQL}
    )
    AND device.idfa IS NOT NULL
    AND device.idfa != ''
    -- 10% deterministic device sample (unbiased rate estimate; for directional comparison only)
    AND MOD(ABS(FARM_FINGERPRINT(device.idfa)), 10) = 0
),

per_device AS (
  SELECT
    bundle,
    os,
    country,
    idfa,
    MAX(CASE WHEN LOWER(event_name) = 'install' THEN 1 ELSE 0 END)   AS has_install,
    MIN(CASE WHEN LOWER(event_name) = 'install' THEN install_at END)  AS install_at,
    MAX(
      CASE
        WHEN LOWER(event_name) IN {LOGIN_EVENTS_SQL}
         AND TIMESTAMP_DIFF(event_ts, install_at, HOUR) BETWEEN 0 AND 24
        THEN 1 ELSE 0
      END
    )                                                                  AS has_d1_login
  FROM unattributed_events
  GROUP BY bundle, os, country, idfa
)

SELECT
  bundle,
  os,
  country,
  COUNT(DISTINCT CASE WHEN has_install = 1 THEN idfa END)                 AS installs_sampled,
  COUNT(DISTINCT CASE WHEN has_install = 1 AND has_d1_login = 1
                      THEN idfa END)                                       AS d1_logins_sampled,
  ROUND(
    SAFE_DIVIDE(
      COUNT(DISTINCT CASE WHEN has_install = 1 AND has_d1_login = 1 THEN idfa END),
      COUNT(DISTINCT CASE WHEN has_install = 1 THEN idfa END)
    ), 4
  )                                                                        AS install_to_login_rate
FROM per_device
WHERE has_install = 1
GROUP BY bundle, os, country
HAVING COUNT(DISTINCT CASE WHEN has_install = 1 THEN idfa END) >= 5
   OR country = 'KOR'
ORDER BY bundle, installs_sampled DESC
"""
df_3 = run_query(query_3, label='Unattributed Install-to-Login Rate (10% sample)')
df_3

✅ Unattributed Install-to-Login Rate (10% sample): 267 rows


,bundle,os,country,installs_sampled,d1_logins_sampled,install_to_login_rate
0,com.netmarble.got,ANDROID,BRA,31828,11346,0.3565
1,com.netmarble.got,ANDROID,USA,11865,4924,0.4150
2,com.netmarble.got,ANDROID,MEX,11361,3120,0.2746
3,com.netmarble.got,ANDROID,IRQ,10019,2550,0.2545
4,com.netmarble.got,ANDROID,TUR,9749,1845,0.1893
...,...,...,...,...,...,...
262,id6499177365,IOS,HUN,5,5,1.0000
263,id6499177365,IOS,GAB,5,5,1.0000
264,id6499177365,IOS,BRN,5,5,1.0000
265,id6499177365,IOS,NZL,5,4,0.8000


## Section 4 - Comparison: Attributed vs Unattributed, by Title \u00d7 OS \u00d7 Country

Side-by-side comparison of attributed vs unattributed install-to-login rate. Output table: attributed rate, unattributed rate, CPI, implied login CPA (= CPI \u00f7 attributed rate), KOR highlighted. Grouped bar chart per title \u00d7 OS.

In [53]:
# Normalize bundle → store_id for df_3 before joining
df_3_base = df_3.copy()
df_3_base['store_id'] = df_3_base['bundle'].str.replace(r'^id(\d+)$', r'\1', regex=True)
df_3_base['install_to_login_rate'] = pd.to_numeric(df_3_base['install_to_login_rate'], errors='coerce')

# Suffix only applies to columns present in BOTH frames — only install_to_login_rate overlaps
df_4 = df_2e.merge(
    df_3_base[['store_id', 'os', 'country', 'installs_sampled', 'd1_logins_sampled', 'install_to_login_rate']],
    on=['store_id', 'os', 'country'],
    suffixes=('_attr', '_unattr'),
    how='left'
)

df_4['i2l_pct_attr']   = (pd.to_numeric(df_4['install_to_login_rate_attr'],   errors='coerce') * 100).round(1)
df_4['i2l_pct_unattr'] = (pd.to_numeric(df_4['install_to_login_rate_unattr'], errors='coerce') * 100).round(1)
df_4['kor_flag']        = df_4['country'] == 'KOR'

# 'installs' and 'd1_logins' keep their names (no suffix — only in df_2e, no conflict)
summary_cols = ['store_id', 'os', 'country',
                'installs', 'i2l_pct_attr',
                'installs_sampled', 'i2l_pct_unattr',
                'cpi_usd', 'implied_login_cpa_usd', 'kor_flag']
display(
    df_4[summary_cols]
    .sort_values(['store_id', 'os', 'installs'], ascending=[True, True, False])
    .reset_index(drop=True)
)

,store_id,os,country,installs,i2l_pct_attr,installs_sampled,i2l_pct_unattr,cpi_usd,implied_login_cpa_usd,kor_flag
0,6469305531,IOS,JPN,2772,98.7,<NA>,NaN,10.01,10.14,False
1,6469305531,IOS,USA,407,95.6,<NA>,NaN,13.67,14.30,False
2,6479595079,IOS,KOR,5192,77.2,201,83.1,9.79,12.68,True
3,6499177365,IOS,JPN,4565,96.4,1045,98.9,5.32,5.52,False
4,6499177365,IOS,KOR,2776,100.0,1057,99.6,18.66,18.67,True
...,...,...,...,...,...,...,...,...,...,...
84,com.netmarble.sololv,ANDROID,AUS,366,87.4,<NA>,NaN,0.45,0.51,False
85,com.netmarble.sololv,ANDROID,MAC,289,81.7,<NA>,NaN,3.51,4.30,False
86,com.netmarble.sololv,ANDROID,CHE,273,83.5,<NA>,NaN,0.38,0.45,False
87,com.netmarble.sololv,ANDROID,NZL,219,83.1,<NA>,NaN,0.45,0.54,False


In [58]:
for (store_id, os_val), grp in df_4.groupby(['store_id', 'os']):
    grp = grp.sort_values('i2l_pct_attr', ascending=False).copy()
    app_name  = TITLE_LOOKUP.get(store_id, store_id)
    title_str = f'{app_name} ({store_id}) — {os_val}'

    # Chart 1: Attributed vs Non-Moloco install-to-login rate
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name=f'Moloco CPI (D{LOGIN_WINDOW_DAYS})',
        x=grp['country'],
        y=grp['i2l_pct_attr'],
        marker_color=['crimson' if c == 'KOR' else '#4C78A8' for c in grp['country']],
        text=grp['i2l_pct_attr'].apply(lambda v: f'{v:.1f}%'),
        textposition='outside',
    ))
    fig.add_trace(go.Bar(
        name='Non-Moloco Baseline',
        x=grp['country'],
        y=grp['i2l_pct_unattr'],
        marker_color=['#FF7F7F' if c == 'KOR' else '#AEC7E8' for c in grp['country']],
        text=grp['i2l_pct_unattr'].apply(
            lambda v: f'{v:.1f}%' if pd.notna(v) else 'N/A'),
        textposition='outside',
    ))
    fig.update_layout(
        title=f'{title_str} — Install-to-Login Rate: Moloco CPI vs Non-Moloco Baseline',
        barmode='group',
        xaxis_title='Country', yaxis_title='Install-to-Login Rate (%)',
        height=450, template='plotly_white',
        legend=dict(orientation='h', y=-0.25)
    )
    fig.write_image(
        os.path.join(CHART_DIR, f'i2l_comparison_{store_id}_{os_val}.png'), scale=2)
    fig.show()

    # Chart 2: Implied login CPA by country (CPI ÷ attributed rate)
    fig2 = go.Figure(go.Bar(
        x=grp['country'],
        y=grp['implied_login_cpa_usd'],
        marker_color=['crimson' if c == 'KOR' else '#4C78A8' for c in grp['country']],
        text=grp['implied_login_cpa_usd'].apply(
            lambda v: f'${v:.2f}' if pd.notna(v) else 'N/A'),
        textposition='outside',
    ))
    fig2.update_layout(
        title=f'{title_str} — Implied Login CPA by Country (CPI ÷ Install-to-Login Rate)',
        xaxis_title='Country', yaxis_title='Implied Login CPA (USD)',
        height=420, template='plotly_white'
    )
    fig2.write_image(
        os.path.join(CHART_DIR, f'implied_login_cpa_{store_id}_{os_val}.png'), scale=2)
    fig2.show()

## Export — HTML (shareable, interactive)

In [59]:
import subprocess, pathlib

nb_file = '260420_cpi_install_to_login_analysis.ipynb'
html_file = nb_file.replace('.ipynb', '.html')

result = subprocess.run(
    [
        'jupyter', 'nbconvert',
        '--to', 'html',
        '--no-input',                          # hide code cells
        '--no-prompt',                         # hide In[]/Out[] prompts
        '--output', html_file,
        nb_file,
    ],
    capture_output=True, text=True,
    cwd=CHART_DIR
)

if result.returncode == 0:
    size_kb = pathlib.Path(CHART_DIR, html_file).stat().st_size // 1024
    print(f'\u2705 Exported: {CHART_DIR}/{html_file} ({size_kb} KB)')
    print('Open in any browser — Plotly charts are fully interactive.')
else:
    print('\u274c nbconvert error:')
    print(result.stderr)

✅ Exported: /Users/haewon.yum/Documents/Queries/premium_support/netmarble/Launch_general/260420_cpi_install_to_login_analysis.html (286 KB)
Open in any browser — Plotly charts are fully interactive.
